# Deutsch Jozsa in native FlagQuantum

Complete local computational Notebook workflow: quantum composition, synthesis, text visualization, seeded sampling from exact named probabilities, and classical decision/recovery. Local drawing and sampling replace cloud services; no hardware result is claimed. Concrete tutorial oracles are specialized; general quantum callable ports are not added.

Algebraically reduce the two concrete predicates to the high bit and its complement; retain the constant pi phase. Classify from zero-outcome probability under the promise rather than the upstream row-count heuristic. Explicit Hadamard transforms replace gate-preparation within/apply.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
import math
from collections import Counter
from random import Random
import flagquantum as fq
from flagquantum.drawer import draw
from flagquantum.synth import (
    PHASE,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
    synthesize,
)

## high bit phase

In [ ]:
@qfunc
def high_bit_phase(angle: float, bit: QBit) -> None:
    PHASE(angle, bit)

## simple

In [ ]:
@qfunc
def simple(x: Output[QArray[QBit]]) -> None:
    allocate(4, x)
    hadamard_transform(x)
    # On four bits, x > 7 is the most significant bit.
    high_bit_phase(math.pi, x[3])
    hadamard_transform(x)

## main

In [ ]:
@qfunc
def main(x: Output[QArray[QBit]]) -> None:
    allocate(3, x)
    hadamard_transform(x)
    # For x in 0..7, max(x ^ 4, x & 3) >= 4 iff the high bit is zero.
    phase(1, math.pi)
    high_bit_phase(-math.pi, x[2])
    hadamard_transform(x)

## classify

In [ ]:
def classify(probabilities: dict[tuple[int, ...], float]) -> str:
    """Apply the promised constant-or-balanced decision to the zero outcome."""
    zero = probabilities.get((0,), 0.0)
    if abs(zero - 1) < 1e-12:
        return "constant"
    if abs(zero) < 1e-12:
        return "balanced"
    raise ValueError("distribution does not certify a constant-or-balanced promise")

## run example

In [ ]:
def run_example() -> None:
    for label, definition in (("simple", simple), ("complex", main)):
        program = synthesize(definition, max_qubits=11)
        print(draw(program.circuit))
        result = fq.run(program)
        probabilities = dict(program.probabilities(result, registers=("x",)))
        print(label, classify(probabilities))
        samples = Random(17).choices(
            list(probabilities), weights=list(probabilities.values()), k=1000
        )
        print("Counts:", dict(Counter(samples)))

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states
from examples.synthesis.algorithms.foundational.deutsch_jozsa.comparisons.circuit import (
    circuit_reference,
    complex_predicate,
)

compare_states(
    synth.synthesize(simple, max_qubits=11),
    circuit_reference(4, lambda value: value > 7),
    ("x",),
)
compare_states(
    synth.synthesize(main, max_qubits=11),
    circuit_reference(3, complex_predicate),
    ("x",),
)
print("Independent full-state comparison: passed")
